# Lecture 3: Declarative modeling and the semantic loss (code companion)

*Discrete Methods for Modern AI*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TuLLI-AI/DiscreteMethodsAI/blob/main/notebooks/lecture3_code.ipynb)

This notebook collects the code examples from the lecture 3 slides, so you can run and change them yourself. It has four parts:

1. **Declarative programs in `symprog`**: the *smokes* program, as a logic program and as ordinary (grounded) logic.
2. **An NLI model**: data from SNLI and a RoBERTa encoder with a classification head.
3. **The semantic loss**: weighted model counting (WMC), the semi-supervised loss in PyTorch, and gradients (numeric and symbolic) in `symprog`.
4. **Circuits and tractability**: the circuit `symprog` builds underneath.

**Setup.** Run the next cell first. On **Google Colab**, `torch`, `transformers` and `sympy` are already installed, so it only adds `symprog` and `datasets` (no GPU needed). On your own machine, also run `pip install torch transformers sympy`.

Part 2 downloads the `roberta-base` model (about 500MB) and the SNLI dataset the first time you run it.

In [ ]:
%pip install -q symprog datasets

---
## Part 1: Declarative programs in `symprog`

A declarative program says **what** we want, not **how** to compute it. Here is a small probabilistic logic program: who smokes, given who is stressed and who influences whom?

Facts carry probabilities (`stress[ann]: 0.8`), rules use variables (`X`, `Y`), and `query` asks for a probability.

In [ ]:
from symprog import Parser

@Parser(language="plp")
def smokes():
    stress[ann]: 0.8
    stress[bob]: 0.4
    influences[ann, bob]: 0.6
    influences[bob, carl]: 0.2

    smokes[X] :- stress[X]
    smokes[X] :- influences[Y, X] & smokes[Y]

    query(smokes[carl])

smokes.inference().inference   ## => 0.1376

### The same program in ordinary logic

**Grounding** fills in `ann`, `bob`, `carl` for `X` and `Y`, which gives propositional logic (lecture 1). Unfolding the grounded rules gives one formula for `smokes_c`, and its probability is a weighted model count.

In [ ]:
@Parser(language="constraint")
def grounded():                    ## a, b, c = ann, bob, carl
    stress_a: 0.8; stress_b: 0.4   ## facts
    infl_ab: 0.6;  infl_bc: 0.2

    ## grounded rules:  smokes_a <-> stress_a
    ##   smokes_b <-> stress_b or (infl_ab and smokes_a)
    ##   smokes_c <-> infl_bc and smokes_b
    ## unfold them into one formula for smokes_c:
    infl_bc and (stress_b or (infl_ab and stress_a))

grounded.inference().inference     ## => 0.1376, the same answer

---
## Part 2: An NLI model

**Natural language inference (NLI)**: given a premise $p$ and a hypothesis $h$, is the relation *entailment*, *contradiction* or *neutral*?

In [ ]:
from datasets import load_dataset

snli = load_dataset("stanfordnlp/snli")      ## Stanford NLI
# mnli = load_dataset("nyu-mll/multi_nli")   ## Multi-genre NLI (larger download)

print(snli["train"][3])
print(snli["train"].features["label"])       ## 0 = entailment, 1 = neutral, 2 = contradiction

A pre-trained **encoder** (RoBERTa) plus a new **classification head** with 3 outputs. The encoder was pre-trained with a *masked language objective*; the head is new and untrained.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification as hf_model
from transformers import AutoTokenizer as hf_tok

MODEL_NAME = "roberta-base"

class NLIModel(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.model = hf_model.from_pretrained(
            model_name,     ## pre-trained encoder
            num_labels=3)   ## + a new head: 3 labels for NLI

    def forward(self, features):      ## forward pass
        output = self.model(**features)
        prob = output.logits.softmax(dim=-1)
        return (output.loss, prob)

nli_model = NLIModel(MODEL_NAME)
tokenizer = hf_tok.from_pretrained(MODEL_NAME)

In [ ]:
p = "Children smiling and waving at camera"
h = "They are smiling at their parents"
batch = tokenizer([(p, h)], return_tensors="pt")
labels = torch.tensor([1])              ## 1 = neutral

print(tokenizer.convert_ids_to_tokens(batch.input_ids[0]))   ## <s> p </s></s> h </s>

loss, prob = nli_model({**batch, "labels": labels})
prob    ## roughly uniform: the head is untrained

---
## Part 3: The semantic loss

### Weighted model counting

The formula $F$: contradiction is **symmetric**, $C(p,h) \leftrightarrow C(h,p)$, plus **label uniqueness** (exactly one label per pair). `E1, C1, N1` are the model's probabilities for $(p, h)$ and `E2, C2, N2` for $(h, p)$. The probability of the formula is its weighted model count, $P_\theta(F) = \textsc{wmc}(F, \theta)$.

In [ ]:
@Parser(language="constraint")
def symmetry():
    E1: 0.3; C1: 0.2; N1: 0.5   ## model on (p, h)
    E2: 0.1; C2: 0.8; N2: 0.1   ## model on (h, p)

    Biconditional(C1, C2)       ## symmetry of C

    E1 or C1 or N1              ## label uniqueness
    not (E1 and C1); not (E1 and N1); not (C1 and N1)
    E2 or C2 or N2
    not (E2 and C2); not (E2 and N2); not (C2 and N2)

symmetry.inference().inference   ## => 0.0598, WMC(F) = P(F)

### Semi-supervised learning in PyTorch

For an **unlabeled** pair, the formula "exactly one label" gives a loss without any gold label: $\mathcal{L}_{\text{sl}} = -\log \textsc{wmc}$. We add it to the ordinary loss on the labeled pair.

In [ ]:
unlabel_p = "John is wearing a red shirt"       ## an unlabeled pair
unlabel_h = "John is wearing a yellow shirt"
batch = tokenizer([(p, h), (unlabel_p, unlabel_h)],
                  return_tensors="pt", padding=True)
labels = torch.tensor([1, -100])        ## -100: no label

label_loss, probs = nli_model({**batch, "labels": labels})

unlab_probs = probs[1:]                 ## the unlabeled pair
wmc = torch.zeros_like(unlab_probs)
for i in range(3):                      ## one world per label
    negate       = 1 - unlab_probs
    negate[:, i] = unlab_probs[:, i]
    wmc[:, i]    = negate.prod(dim=-1)

wmc_loss = -torch.log(wmc.sum(dim=-1)).mean()

loss_weight = 0.3                       ## lambda
loss = label_loss + loss_weight * wmc_loss
loss.backward()                         ## gradients reach the whole model
print(label_loss, wmc_loss)

### The toy example, with gradients

The same computation on fixed probabilities $(0.3, 0.2, 0.5)$ for $(E, C, N)$. PyTorch records a **computation graph** and `backward()` runs the chain rule over it.

In [ ]:
unlab_probs = torch.tensor([[0.3, 0.2, 0.5]],   ## our toy example
                           requires_grad=True)
wmc = torch.zeros_like(unlab_probs)
for i in range(3):
    negate       = 1 - unlab_probs
    negate[:, i] = unlab_probs[:, i]
    wmc[:, i]    = negate.prod(dim=-1)
wmc_loss = -torch.log(wmc.sum(dim=-1)).mean()

print(wmc)          ## => tensor([[0.12, 0.07, 0.28]], grad_fn=<CopySlices>)
print(wmc_loss)     ## => tensor(0.7550, grad_fn=<NegBackward0>), -log 0.47

wmc_loss.backward()       ## backpropagate
unlab_probs.grad          ## => tensor([[ 0.21,  0.32, -0.38]])

### Gradients in `symprog`

`symprog` computes gradients too. Compare them with PyTorch's: what is different, and why? (See the exercises.)

In [ ]:
@Parser(language="constraint")
def program():
    E: 0.3; C: 0.2; N: 0.5             ## our toy example
    E or C or N                        ## label uniqueness
    not (E and C); not (E and N); not (C and N)

print(program.inference().inference)   ## => 0.47
program.grad()                         ## gradients of P(F)

### Under the hood: symbolic gradients

`symprog` uses **computer algebra** (SymPy), so you can ask for the formula instead of a number.

In [ ]:
import sympy

P = program.inference(symbolic=True).inference
P

In [ ]:
sympy.expand(P)              ## linear in each weight

In [ ]:
grad = program.grad(symbolic=True)["weights"]
grad                         ## dP/dC, dP/dE, dP/dN

In [ ]:
E, C, N = sympy.symbols("E C N", positive=True)   ## symprog's symbols are positive
grad.subs({E: 0.3, C: 0.2, N: 0.5}).T             ## the numbers from before

---
## Part 4: Circuits and tractability

Computing WMC is \#P-hard in general. `symprog` first **compiles** a formula into a circuit (an SDD), then evaluates it in one pass.

In [ ]:
@Parser(language="constraint")
def program():
    a: 0.4; b: 0.7
    a or b

program.inference().inference           ## => 0.82

In [ ]:
program.inference(mode="Formula").inference      ## the compiled circuit, e.g. a | (b & ~a)

In [ ]:
program.inference(symbolic=True).inference       ## as arithmetic, e.g. a + b*(1 - a)

The circuit is **deterministic**: its two branches cannot both be true, so WMC is one bottom-up pass, e.g. $0.4 + 0.7 \cdot (1 - 0.4) = 0.82$. (The variable order, and so the exact circuit, can differ from the slides; it depends on what was compiled before.)

To draw it, we use Graphviz. It is already installed on Colab; on your own machine you need `pip install graphviz` plus the Graphviz program (e.g. `brew install graphviz` or `apt install graphviz`).

In [ ]:
dot = program.solver.formula.dot        ## the circuit in the DOT graph language
print(dot)

try:                                    ## draw it, if Graphviz is installed
    import graphviz
    from IPython.display import SVG, display
    display(SVG(graphviz.Source(dot).pipe(format="svg").decode()))
except Exception as err:
    print("(Graphviz is not available here:", type(err).__name__ + ")")

---
## Exercises

1. **Gradients.** PyTorch gave `[0.21, 0.32, -0.38]` for $(E, C, N)$, but `symprog`'s `program.grad()` gives different numbers. Explain the difference, and turn `symprog`'s gradients into PyTorch's. *(Hint: what is each one the gradient of, and in which order are the weights?)*
2. **Spurious learning.** In the `symmetry` program, try a model that predicts $C$ for both orders, and one that never predicts $C$. Which one does the rule prefer?
3. **Your own rule.** Write the rule $E(p,h) \rightarrow \neg C(h,p)$ (with uniqueness) and compute its probability for the weights above.
4. **Circuits.** Print the compiled circuit (`mode="Formula"`) for the uniqueness constraint `E or C or N` plus the three `not (... and ...)` clauses. Is it deterministic?